# Memoripy + OpenAI live notebook

This notebook is for interactive, real-API testing of the v3 memory flow.

What it does:
- creates a file-backed `MemoryClient`
- uses OpenAI for chat and embeddings
- captures memories and tool results
- lets you ask live questions against stored memory
- shows you the `memory_pack` that grounded the answer

Start Jupyter from the repo root if possible:

```bash
cd /Users/khazarayaz/Desktop/memoripy/memoripy
jupyter lab
```

In [1]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "memoripy").exists():
        sys.path.insert(0, str(candidate))
        print(f"Using repo root: {candidate.resolve()}")
        break
else:
    raise RuntimeError("Start Jupyter from the repo root or its examples/ directory.")

Using repo root: /Users/khazarayaz/Desktop/memoripy/memoripy


In [2]:
from pprint import pprint
import os
import shutil

from memoripy import MemoryClient, OpenAIChatModel, OpenAIEmbeddingModel

In [ ]:
# Paste your key here or export OPENAI_API_KEY before starting Jupyter.
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or ""

# Use the model IDs available in your account.
CHAT_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

STORE_DIR = Path(".memoripy-openai-live")
USER_ID = "notebook-user"
AGENT_ID = "jarvis"
RUN_ID = "notebook-session-1"

# Set to True if you want a fresh store each time you rerun setup.
RESET_STORE = False

In [5]:
if RESET_STORE and STORE_DIR.exists():
    shutil.rmtree(STORE_DIR)

client = MemoryClient.from_path(
    STORE_DIR,
    chat_model=OpenAIChatModel(api_key=OPENAI_API_KEY, model_name=CHAT_MODEL),
    embedding_model=OpenAIEmbeddingModel(api_key=OPENAI_API_KEY, model_name=EMBEDDING_MODEL),
)

print("Store:", STORE_DIR.resolve())

Store: /Users/khazarayaz/Desktop/memoripy/memoripy/examples/.memoripy-openai-live


In [6]:
def show_pack(memory_pack: dict) -> None:
    print("\nIntent:", memory_pack["intent"])
    print("\nProfile:")
    pprint(memory_pack["profile"])
    print("\nPreferences:")
    pprint(memory_pack["preferences"])
    print("\nRelationships:")
    pprint(memory_pack["relationships"])
    print("\nRecent episodes:")
    pprint(memory_pack["recent_episodes"])
    print("\nTool observations:")
    pprint(memory_pack["tool_observations"])
    print("\nCitations:")
    pprint(memory_pack["citations"])


def seed_demo_memories() -> dict:
    return client.capture(
        messages=[
            {"role": "user", "content": "My name is Khazar"},
            {"role": "user", "content": "I live in Istanbul"},
            {"role": "user", "content": "My favorite city is Tokyo"},
            {"role": "assistant", "content": "I will remember that."},
        ],
        events=[
            {
                "event_type": "tool_result",
                "name": "calendar.lookup",
                "content": "Dinner with Mert is tomorrow at 7 PM",
            }
        ],
        user_id=USER_ID,
        agent_id=AGENT_ID,
        run_id=RUN_ID,
        idempotency_key="notebook-seed-1",
    )


def ask(prompt: str, tool_events: list[dict] | None = None, store: bool = True, context_policy: str = "balanced") -> dict:
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": prompt}],
        user_id=USER_ID,
        agent_id=AGENT_ID,
        run_id=RUN_ID,
        memory_strategy="v3",
        include_memory_pack=True,
        context_policy=context_policy,
        tool_events=tool_events or [],
        store=store,
    )

    print("USER:", prompt)
    print("\nASSISTANT:\n")
    print(response["choices"][0]["message"]["content"])
    print("\nTop memory hits:", len(response["memory"]["results"]))
    show_pack(response["memory_pack"])
    return response


def show_all_memories() -> None:
    payload = client.get_all(user_id=USER_ID, agent_id=AGENT_ID, run_id=RUN_ID)
    print(f"Stored memories: {len(payload['results'])}")
    for index, item in enumerate(payload["results"], start=1):
        memory = item["memory"]
        print(
            f"{index}. kind={memory['kind']} key={memory['key']} value={memory['value']} state={memory['state']}"
        )

## 1. Seed a few facts and one tool result

In [7]:
seed_result = seed_demo_memories()
pprint(seed_result)

{'created': [{'record_id': 'memory_9967029eb2374841a431bcc09d6bd2d9',
              'summary': 'Name: Khazar',
              'version_id': 'version_1a3ae6dd192c43c29700cb3e23bdf34a'},
             {'record_id': 'memory_da130e5209df4ccfb24abc2ee1ade789',
              'summary': 'Location: Istanbul',
              'version_id': 'version_5d21ee34967b474081478c1224da419f'},
             {'record_id': 'memory_67ef8ee6cf32420ba462f30fcf61befd',
              'summary': 'Favorite city: Tokyo',
              'version_id': 'version_b3eefe2579844e599d3738f30b498b85'},
             {'record_id': 'memory_1a1f4d6a609d43358bd6599acd1ede87',
              'summary': 'Tool result: Dinner with Mert is tomorrow at 7 PM',
              'version_id': 'version_7f11c111c62c4f0fa2aa394cd5285215'}],
 'episodic_memory_ids': ['memory_1a1f4d6a609d43358bd6599acd1ede87'],
 'evidence_ids': ['evidence_a19a9e1f8a5e38790368f889',
                  'evidence_a956bbc826a10d3bdde437fa',
                  'evidence_48703

## 2. Inspect the raw v3 context pack directly

In [8]:
pack = client.context.build(
    query="What do you remember about me and what is on my calendar?",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    run_id=RUN_ID,
    include_debug=True,
)

pprint(pack.to_dict())

{'citations': [{'event_type': 'message',
                'evidence_id': 'evidence_a19a9e1f8a5e38790368f889',
                'memory_id': 'memory_9967029eb2374841a431bcc09d6bd2d9',
                'occurred_at': '2026-03-18T03:36:19.306162Z',
                'scope': {'agent_id': 'jarvis',
                          'run_id': 'notebook-session-1',
                          'user_id': 'notebook-user'},
                'source_type': 'message',
                'summary': 'My name is Khazar',
                'text': 'My name is Khazar'},
               {'event_type': 'message',
                'evidence_id': 'evidence_4870312ded0d86a98a5217c1',
                'memory_id': 'memory_67ef8ee6cf32420ba462f30fcf61befd',
                'occurred_at': '2026-03-18T03:36:19.306207Z',
                'scope': {'agent_id': 'jarvis',
                          'run_id': 'notebook-session-1',
                          'user_id': 'notebook-user'},
                'source_type': 'message',
              

## 3. Ask a live question through OpenAI

Change `prompt` and rerun this cell as much as you want.

In [9]:
prompt = "What do you remember about me?"
response = ask(prompt)

USER: What do you remember about me?

ASSISTANT:

I remember your name is Khazar, and you're located in Istanbul. Your favorite city is Tokyo. If there's anything else you'd like to add or update, feel free to let me know!

Top memory hits: 4

Intent: general

Profile:
[{'citation_evidence_ids': ['evidence_a19a9e1f8a5e38790368f889'],
  'citations': [{'event_type': 'message',
                 'evidence_id': 'evidence_a19a9e1f8a5e38790368f889',
                 'occurred_at': '2026-03-18T03:36:19.306162Z',
                 'source_type': 'message',
                 'summary': 'My name is Khazar'}],
  'confirmation_count': 1,
  'key': 'name',
  'kind': 'profile_attribute',
  'layer': 'semantic',
  'memory_id': 'memory_9967029eb2374841a431bcc09d6bd2d9',
  'rank_breakdown': {'access': 0.0,
                     'graph': 0.1,
                     'intent': 0.5,
                     'lexical': 0.0,
                     'llm': 0.0,
                     'recency': 0.9998264252838753,
           

## 4. Test a live tool result in the same turn

In [10]:
tool_events = [
    {
        "event_type": "tool_result",
        "name": "weather.lookup",
        "content": "Tomorrow in Istanbul it will be sunny and 21 C",
    }
]

response = ask("Should I take sunglasses tomorrow?", tool_events=tool_events)

USER: Should I take sunglasses tomorrow?

ASSISTANT:

Yes, since it will be sunny tomorrow in Istanbul with a temperature of 21°C, it's a good idea to take sunglasses with you.

Top memory hits: 1

Intent: general

Profile:
[]

Preferences:
[]

Relationships:
[]

Recent episodes:
[]

Tool observations:
[{'citation_evidence_ids': ['evidence_a4477f4483e242376c5de330'],
  'citations': [{'event_type': 'tool_result',
                 'evidence_id': 'evidence_a4477f4483e242376c5de330',
                 'occurred_at': '2026-03-18T03:36:19.306226Z',
                 'source_type': 'tool_result',
                 'summary': 'Dinner with Mert is tomorrow at 7 PM'}],
  'confirmation_count': 1,
  'key': 'episode_24eab03c678b34c9',
  'kind': 'episodic_summary',
  'layer': 'episodic',
  'memory_id': 'memory_1a1f4d6a609d43358bd6599acd1ede87',
  'rank_breakdown': {'access': 0.0,
                     'graph': 0.0,
                     'intent': 0.35,
                     'lexical': 0.25,
              

## 5. Change a fact and verify recall updates

In [11]:
client.capture(
    messages=[{"role": "user", "content": "I moved to Berlin last month"}],
    user_id=USER_ID,
    agent_id=AGENT_ID,
    run_id=RUN_ID,
    idempotency_key="notebook-move-1",
)

response = ask("Where do I live now?")

USER: Where do I live now?

ASSISTANT:

You were in Berlin last month, but I don't have current information on your whereabouts. If you're still in Berlin or have moved, please let me know!

Top memory hits: 5

Intent: profile

Profile:
[{'citation_evidence_ids': ['evidence_a956bbc826a10d3bdde437fa',
                            'evidence_fcb654c77c1fcdd186943dfc'],
  'citations': [{'event_type': 'message',
                 'evidence_id': 'evidence_a956bbc826a10d3bdde437fa',
                 'occurred_at': '2026-03-18T03:36:19.306194Z',
                 'source_type': 'message',
                 'summary': 'I live in Istanbul'},
                {'event_type': 'message',
                 'evidence_id': 'evidence_fcb654c77c1fcdd186943dfc',
                 'occurred_at': '2026-03-18T03:37:19.607656Z',
                 'source_type': 'message',
                 'summary': 'I moved to Berlin last month'}],
  'confirmation_count': 1,
  'key': 'location',
  'kind': 'profile_attribute',
  'lay

## 6. Inspect everything stored so far

In [12]:
show_all_memories()

snapshot = client.export()
print("\nEvidence count:", len(snapshot["evidence"]))
print("Version count:", len(snapshot["versions"]))

Stored memories: 9
1. kind=episodic_summary key=episode_41228d783d6b4702 value=You were in Berlin last month, but I don't have current information on your whereabouts. If you're still in Berlin or have moved, please let me know! state=active
2. kind=preference key=avoids_6b260ef6eb4c value=have current information on your whereabouts state=active
3. kind=profile_attribute key=location value=Berlin last month state=active
4. kind=episodic_summary key=episode_b52824c3c2ccc260 value=Tomorrow in Istanbul it will be sunny and 21 C state=active
5. kind=episodic_summary key=episode_83ae0eb5b68ace07 value=Yes, since it will be sunny tomorrow in Istanbul with a temperature of 21°C, it's a good idea to take sunglasses with you. state=active
6. kind=episodic_summary key=episode_7961c380bd5616b5 value=I remember your name is Khazar, and you're located in Istanbul. Your favorite city is Tokyo. If there's anything else you'd like to add or update, feel free to let me know! state=active
7. kind=episo